# Big coding · LoRA supervised fine-tuning

**Goal:** implement low-rank adapters and completion-only SFT on a tiny causal language model. **Time:** 90–120 minutes. **Prerequisites:** self-attention, autograd, cross-entropy, basic training loops. **Runtime:** CPU + PyTorch. No external datasets, APIs, or model downloads.

This is a controlled mechanics exercise, not a pretrained large-model benchmark. We first pretrain a tiny decoder on synthetic token patterns, then adapt it to a different synthetic prompt/completion task. Improvements on training examples do not demonstrate real-world instruction following.

Save a copy in Drive. Complete the TODO functions before running student tests. Sample solution cells are folded by default; use **Show code** to expand them.

## 1. Problem and contract

Implement three pieces:

1. `LoRALinear(base, rank, alpha)`: freeze the base linear layer and learn `A: [rank, in_features]`, `B: [out_features, rank]`. Compute `base(x) + (alpha/rank) * x @ A.T @ B.T`. Initialize A randomly and B to zero.
2. `make_batch(examples)`: each example is `(prompt_ids, completion_ids)`. Append EOS, right-pad, shift inputs and labels for next-token prediction. Mask prompt and padded **target positions** with `-100`; the first completion token must be supervised from the final prompt position.
3. `sft_loss(logits, labels)`: mean cross-entropy over all supervised tokens, with `ignore_index=-100`. For this task every batch contains at least one supervised token.

Use the supplied model/training loop to train only adapters in the Q and V projections. Verify base weights remain unchanged, adapter gradients flow, and completion loss falls. Then save and reload adapter tensors onto the same base checkpoint.


In [ ]:
import copy
import math
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(23)
torch.set_num_threads(1)
PAD, BOS, SEP, EOS = 0, 1, 2, 3
VOCAB, D, HEADS, MAX_LEN = 16, 32, 4, 16

class TinyLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D)
        self.pos = nn.Embedding(MAX_LEN, D)
        self.q, self.k, self.v, self.o = [nn.Linear(D, D, bias=False) for _ in range(4)]
        self.norm = nn.LayerNorm(D)
        self.ff = nn.Sequential(nn.Linear(D, 2*D), nn.GELU(), nn.Linear(2*D, D))
        self.norm2 = nn.LayerNorm(D)
        self.head = nn.Linear(D, VOCAB, bias=False)

    def forward(self, ids):
        b, t = ids.shape
        x = self.embed(ids) + self.pos(torch.arange(t, device=ids.device))
        z = self.norm(x)
        q, k, v = [layer(z).reshape(b, t, HEADS, D//HEADS).transpose(1, 2)
                   for layer in (self.q, self.k, self.v)]
        context = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
        x = x + self.o(context.transpose(1, 2).contiguous().reshape(b, t, D))
        return self.head(x + self.ff(self.norm2(x)))

# Pretrain on cyclic token sequences; this yields a reproducible local base checkpoint.
base = TinyLM()
pretrain_optimizer = torch.optim.AdamW(base.parameters(), lr=0.01)
for step in range(100):
    start = torch.randint(0, 8, (32, 1))
    ids = 4 + (start + torch.arange(9)) % 8
    loss = F.cross_entropy(base(ids[:, :-1]).reshape(-1, VOCAB), ids[:, 1:].reshape(-1))
    pretrain_optimizer.zero_grad()
    loss.backward()
    pretrain_optimizer.step()
base.eval()
base_state = copy.deepcopy(base.state_dict())
print('Pretraining final loss:', round(loss.item(), 4))

# New task: map each prompt token to its duplicated successor token, then EOS.
# Variable prompt lengths exercise completion boundaries and right-padding.
examples = [([BOS] + [SEP]*(i%2) + [4+i, SEP], [4+(i+1)%8]*2) for i in range(8)]


## 2. Your implementation

The labels are aligned with input positions, not the original sequence positions. If the prompt length is P, completion supervision starts at label index P−1 after shifting. Never mask by token value alone: use sequence lengths and prompt boundaries.

All training inputs use right-padding, so a real token cannot attend to later padding under the causal mask. This simplification would not handle left-padding or packed sequences without additional masks.


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank=4, alpha=8):
        super().__init__()
        # TODO: retain/freeze base; create trainable A and zero-initialized B; set scaling.
        raise NotImplementedError('Implement LoRALinear')

    def forward(self, x):
        # TODO: frozen base result plus scaled low-rank update.
        raise NotImplementedError

def make_batch(examples):
    # TODO: append EOS, shift, pad inputs with PAD, and mask non-completion labels with -100.
    raise NotImplementedError

def sft_loss(logits, labels):
    # TODO: flatten logits/labels and average over valid target tokens.
    raise NotImplementedError


## 3. Unit tests

These tests cover zero-update initialization, shape/scale, trainable parameter count, prompt-boundary masking, padding, and the ignored-token loss denominator. At initialization A may receive a zero gradient because B starts at zero; after B updates, A should receive a useful gradient too.


In [ ]:
def check_parts(layer_cls, batch_fn, loss_fn):
    torch.manual_seed(31)
    linear = nn.Linear(6, 5, bias=True)
    original = copy.deepcopy(linear)
    layer = layer_cls(linear, rank=2, alpha=4)
    inputs = torch.randn(2, 3, 6)
    torch.testing.assert_close(layer(inputs), original(inputs))
    assert not any(p.requires_grad for p in layer.base.parameters())
    assert sum(p.numel() for p in layer.parameters() if p.requires_grad) == 2*(6+5)
    with torch.no_grad():
        layer.A.fill_(0.2)
        layer.B.fill_(0.3)
    torch.testing.assert_close(layer(inputs), original(inputs) + 2 * inputs @ layer.A.T @ layer.B.T)
    ids, labels = batch_fn([([BOS, 4, SEP], [7, 8]), ([BOS, SEP], [6])])
    assert ids.tolist() == [[BOS, 4, SEP, 7, 8], [BOS, SEP, 6, PAD, PAD]]
    assert labels.tolist() == [[-100, -100, 7, 8, EOS], [-100, 6, EOS, -100, -100]]
    logits = torch.randn(2, 5, VOCAB)
    valid = labels != -100
    expected = F.cross_entropy(logits[valid], labels[valid])
    torch.testing.assert_close(loss_fn(logits, labels), expected)
    altered = logits.clone()
    altered[~valid] = 100
    torch.testing.assert_close(loss_fn(altered, labels), expected)
    print('PASS: LoRA shape/scaling/freezing, batch boundaries, padding, completion loss.')

check_parts(LoRALinear, make_batch, sft_loss)


## 4. Sample solution — expand when ready

The annotated reference implements the same contracts. It uses separate names so your TODO implementation remains intact.


In [ ]:
#@title Sample solution — expand to review { display-mode: "form" }
class ReferenceLoRALinear(nn.Module):
    def __init__(self, base, rank=4, alpha=8):
        super().__init__()  # Initialize PyTorch's module/parameter registration.
        self.base = base  # Retain the existing pretrained linear weights and bias.
        self.base.requires_grad_(False)  # Prevent base gradients and optimizer updates.
        self.scale = alpha / rank  # Apply the standard LoRA scaling convention.
        self.A = nn.Parameter(base.weight.new_empty(rank, base.in_features))  # Down-project features.
        self.B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))  # Zero update initially.
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # Random A lets B learn on the first step.

    def forward(self, x):
        update = (x @ self.A.T) @ self.B.T  # Map [..., in] -> [..., rank] -> [..., out].
        return self.base(x) + self.scale * update  # Add only the scaled adapter contribution.

def reference_make_batch(examples):
    sequences = [prompt + completion + [EOS] for prompt, completion in examples]  # Include an end target.
    width = max(len(seq)-1 for seq in sequences)  # Shifted sequence lengths determine padding width.
    ids = torch.full((len(examples), width), PAD, dtype=torch.long)  # Initialize right-padded inputs.
    labels = torch.full_like(ids, -100)  # Default to excluded targets, including all padding.
    for row, ((prompt, _), seq) in enumerate(zip(examples, sequences)):
        length = len(seq)-1  # Each nonfinal token has a next-token prediction target.
        ids[row, :length] = torch.tensor(seq[:-1])  # Inputs omit the final EOS token.
        labels[row, :length] = torch.tensor(seq[1:])  # Targets are shifted one place left.
        labels[row, :len(prompt)-1] = -100  # Ignore prompt targets but keep the first completion token.
    return ids, labels  # Both tensors use shape [batch, shifted padded length].

def reference_sft_loss(logits, labels):
    # Cross-entropy accepts raw logits; ignored labels contribute neither loss nor denominator.
    return F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), ignore_index=-100)

check_parts(ReferenceLoRALinear, reference_make_batch, reference_sft_loss)  # Run the student contract tests.


## 5. SFT experiment and adapter round trip

The runnable experiment defaults to the reference implementation. Once your functions pass the tests, change the aliases in the next cell to your implementations. Only Q/V adapters train; every original parameter stays frozen. The loss check measures fit to eight training examples, not generalization. A separate held-out evaluation is an extension.


In [ ]:
Layer, batch_fn, loss_fn = ReferenceLoRALinear, reference_make_batch, reference_sft_loss
# To evaluate your own solution instead, use: Layer, batch_fn, loss_fn = LoRALinear, make_batch, sft_loss

def attach_adapters(model):
    model.requires_grad_(False)
    model.q = Layer(model.q, rank=4, alpha=8)
    model.v = Layer(model.v, rank=4, alpha=8)
    return model

torch.manual_seed(41)
model = attach_adapters(copy.deepcopy(base))
frozen_before = {name: p.detach().clone() for name, p in model.named_parameters() if not p.requires_grad}
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=0.03, weight_decay=0.0)
ids, labels = batch_fn(examples)
with torch.no_grad():
    initial = loss_fn(model(ids), labels).item()
model.train()
for step in range(250):
    loss = loss_fn(model(ids), labels)
    optimizer.zero_grad()
    loss.backward()
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in trainable)
    optimizer.step()
model.eval()
with torch.no_grad():
    final = loss_fn(model(ids), labels).item()
assert final < initial * 0.75, (initial, final)
for name, p in model.named_parameters():
    if name in frozen_before:
        assert torch.equal(p, frozen_before[name]), name
assert any(p.grad.abs().sum() > 0 for p in trainable)

# Save only adapter matrices; reconstruct using the exact same base checkpoint and config.
adapter = {name: p.detach().clone() for name, p in model.named_parameters() if p.requires_grad}
checkpoint = {'rank': 4, 'alpha': 8, 'targets': ['q', 'v'], 'state': adapter}
torch.save(checkpoint, 'lora-adapter.pt')
loaded = torch.load('lora-adapter.pt', weights_only=True)
assert (loaded['rank'], loaded['alpha'], loaded['targets']) == (4, 8, ['q', 'v'])
restored_base = TinyLM()
restored_base.load_state_dict(base_state)
restored = attach_adapters(restored_base).eval()
result = restored.load_state_dict(loaded['state'], strict=False)
assert not result.unexpected_keys
assert all(not key.endswith(('.A', '.B')) for key in result.missing_keys)
with torch.no_grad():
    torch.testing.assert_close(model(ids), restored(ids))
print(f'Completion training loss: {initial:.4f} -> {final:.4f}')
print('Trainable adapter parameters:', sum(p.numel() for p in trainable))
print('PASS: loss improvement, frozen base, finite adapter gradients, saved adapter round trip.')


## 6. Review and extensions

1. Why initialize B to zero but not both A and B to zero?
2. Why must prompt masking start from shifted target positions? Which input predicts the first completion token?
3. Why does LoRA reduce optimizer state without eliminating base-weight or activation memory?
4. Add held-out prompt/completion pairs and compare training and validation loss.
5. Sweep rank and alpha, then try adapting Q only, V only, or both.
6. Merge `scale * B @ A` into base weights and verify output equivalence.
7. Port the pipeline to a pretrained model and real tokenizer; inspect special tokens, template boundaries, and truncation before training.

Source: [LoRA paper](https://arxiv.org/abs/2106.09685). This notebook implements the adapter equation directly rather than requiring a PEFT library.
